In [2]:
%pip install webdriver-manager selenium

Note: you may need to restart the kernel to use updated packages.


In [14]:
%conda install lxml

Channels:
 - conda-forge
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: C:\Users\Jesse\miniforge3

  added / updated specs:
    - lxml


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    lxml-6.0.2                 |  py312h2f35c63_2         1.2 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         1.2 MB

The following NEW packages will be INSTALLED:

  lxml               conda-forge/win-64::lxml-6.0.2-py312h2f35c63_2 



lxml-6.0.2           | 1.2 MB    |            |   0% 
lxml-6.0.2           | 1.2 MB    | 1          |   1% 
lxml-6.0.2           | 1.2 MB    | ########   |  81% 
lxml-6.0.2           | 1.2 MB    | ########## | 100% 
lxml-6.0.2           | 1.2 MB    | ########## | 100% 
                                                     
 done
Preparing transactio

In [15]:
import time
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

path_to_driver = r"C:\Users\Jesse\OneDrive\Documents\msedgedriver.exe"
service = Service(executable_path=path_to_driver)
driver = webdriver.Edge(service=service)

BASE_URL = "https://humantraffickinghotline.org/en/statistics"
years = ["2024","2023","2022","2021","2020","2019","2018","2017","2016","2015"]
all_year_tables = []

def get_visible_table(driver):
    """Return the first table that is actually visible on screen."""
    tables = driver.find_elements(By.CSS_SELECTOR, ".nhth-states-table--wrapper table")
    for table in tables:
        if table.is_displayed() and table.text.strip() != "":
            return table
    return None

try:
    driver.get(BASE_URL)
    driver.maximize_window()
    time.sleep(6)

    for year in years:
        print(f"\n--- Processing {year} ---")

        # Click the year tab
        links = driver.find_elements(By.TAG_NAME, "a")
        target_link = next((l for l in links if l.text.strip() == year), None)

        if not target_link:
            print(f"  Year {year} link not found.")
            continue

        driver.execute_script("arguments[0].click();", target_link)

        # Wait for a visible, non-empty table to appear
        try:
            WebDriverWait(driver, 15).until(
                lambda d: get_visible_table(d) is not None
            )
        except:
            print(f"  Timed out waiting for visible table for {year}.")
            continue

        time.sleep(1)
        table = get_visible_table(driver)

        if table is None:
            print(f"  No visible table found for {year}.")
            continue

        try:
            html = table.get_attribute("outerHTML")
            df = pd.read_html(StringIO(html))[0]
            if len(df) > 0:
                df["Year"] = year
                all_year_tables.append(df)
                print(f"  Success! Captured {len(df)} rows for {year}.")
            else:
                print(f"  Table was visible but empty for {year}.")
        except Exception as e:
            print(f"  Error parsing table for {year}: {e}")

    if all_year_tables:
        final_df = pd.concat(all_year_tables, ignore_index=True)
        final_df.to_csv("trafficking_states_master.csv", index=False)
        print(f"\n--- ALL DONE --- Total rows: {len(final_df)}")
    else:
        print("No tables collected.")

finally:
    driver.quit()


--- Processing 2024 ---
  Success! Captured 55 rows for 2024.

--- Processing 2023 ---
  Success! Captured 55 rows for 2023.

--- Processing 2022 ---
  Success! Captured 54 rows for 2022.

--- Processing 2021 ---
  Success! Captured 55 rows for 2021.

--- Processing 2020 ---
  Success! Captured 55 rows for 2020.

--- Processing 2019 ---
  Success! Captured 55 rows for 2019.

--- Processing 2018 ---
  Success! Captured 54 rows for 2018.

--- Processing 2017 ---
  Success! Captured 53 rows for 2017.

--- Processing 2016 ---
  Success! Captured 55 rows for 2016.

--- Processing 2015 ---
  Success! Captured 55 rows for 2015.

--- ALL DONE --- Total rows: 546
